### Ulta top10 브랜드 메타 크롤링 (200개 크롤링 -> 최근 3개월 필터링)

In [9]:
from playwright.async_api import async_playwright
from datetime import datetime, timedelta
from urllib.parse import quote
from pathlib import Path
import pandas as pd
import asyncio
import threading
import json
import re

INPUT_JSON = "./ulta_rankings_current.jsonl"
PAGE_ID_JSON = "./brand_page_ids_all.json"

HEADLESS = False
MAX_ADS_PER_BRAND = 200
SCROLL_ROUNDS = 40
SCROLL_WAIT_MS = 2200

RAW_OUTPUT = "./meta_crawling_result/ulta_meta.csv"
RAW_90D_OUTPUT = "./meta_crawling_result/ulta_meta_90.csv"
SUMMARY_90D_OUTPUT = "./meta_crawling_result/ulta_meta_summary.csv"


def load_product_json(path):
    ext = Path(path).suffix.lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext == ".jsonl":
            data = [json.loads(line) for line in f if line.strip()]
        else:
            data = json.load(f)
    return data


def extract_unique_brands(data):
    return list(dict.fromkeys([str(x.get("brand", "")).strip() for x in data if x.get("brand")]))


def detect_channel(path):
    name = Path(path).name.lower()
    for ch in ["ulta", "sephora", "rakuten", "qoo10"]:
        if ch in name:
            return ch
    raise ValueError("채널 감지 실패")


def load_channel_page_map(path, channel):
    config = json.load(open(path, "r", encoding="utf-8"))
    return config[channel]["market"], config[channel]["brands"]


def build_page_url(page_id, country):
    return f"https://www.facebook.com/ads/library/?active_status=active&ad_type=all&country=ALL&is_targeted_country=false&media_type=all&search_type=page&view_all_page_id={page_id}"



def normalize_text(text):
    return re.sub(r"\s+", " ", text.lower().strip()) if text else ""


def extract_date(text):
    m = re.search(r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})", text)
    if m:
        return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    return ""


def extract_text(raw):
    if not raw:
        return ""
    lines = [x.strip() for x in raw.split("\n") if x.strip()]
    for line in lines:
        if len(line) > 30 and "게재 시작" not in line and "광고" not in line:
            return line
    return ""


async def scroll(page):
    prev = 0
    for _ in range(SCROLL_ROUNDS):
        h = await page.evaluate("document.body.scrollHeight")
        if h == prev:
            break
        prev = h
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(SCROLL_WAIT_MS)


async def get_cards(page):
    return page.locator('[data-testid="ad-library-dynamic-content-container"]').locator("xpath=ancestor::div[5]")


async def parse_card(card, brand, page_id, market, retailer, mode):
    raw = ""
    try:
        raw = await card.inner_text()
    except:
        pass

    return {
        "brand": brand,
        "page_id": page_id,
        "market": market,
        "retailer": retailer,
        "source_mode": mode,
        "library_id": re.search(r"\d{10,}", raw).group(0) if re.search(r"\d{10,}", raw) else "",
        "media_type": "video" if "video" in raw.lower() else "image",
        "start_date": extract_date(raw),
        "ad_text": extract_text(raw),
    }


async def collect(page, brand, page_id, market, retailer, mode):
    await scroll(page)

    cards = await get_cards(page)
    count = await cards.count()

    ads = []
    seen = set()

    for i in range(count):
        card = cards.nth(i)
        data = await parse_card(card, brand, page_id, market, retailer, mode)

        key = (data["library_id"], normalize_text(data["ad_text"]))
        if key in seen:
            continue

        seen.add(key)
        ads.append(data)

        if len(ads) >= MAX_ADS_PER_BRAND:
            break

    return ads


def summarize(df):
    rows = []
    now = datetime.now()

    for brand, g in df.groupby("brand"):
        total = len(g)
        img = (g["media_type"] == "image").sum()
        vid = (g["media_type"] == "video").sum()

        dates = pd.to_datetime(g["start_date"], errors="coerce").dropna()

        latest = dates.max().strftime("%Y-%m-%d") if len(dates) else ""
        recent30 = (dates >= pd.Timestamp(now - timedelta(days=30))).sum()

        rows.append({
            "brand": brand,
            "total_ads": total,
            "image_ads": int(img),
            "video_ads": int(vid),
            "image_ratio": round(img/total, 3) if total else 0,
            "video_ratio": round(vid/total, 3) if total else 0,
            "latest_ad_date": latest,
            "recent_30d_ads": int(recent30),
        })

    return pd.DataFrame(rows)


async def main():
    product = load_product_json(INPUT_JSON)
    brands = extract_unique_brands(product)

    channel = detect_channel(INPUT_JSON)
    market, page_map = load_channel_page_map(PAGE_ID_JSON, channel)

    all_ads = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()

        for brand in brands:
            print(f"[START] {brand}")

            page_id = page_map[brand]["page_id"]

            await page.goto(build_page_url(page_id, market))
            await page.wait_for_timeout(3000)

            ads = await collect(page, brand, page_id, market, channel, "page")

            all_ads.extend(ads)
            print(f"[DONE] {brand}: {len(ads)}")

        await browser.close()

    # ====================
    # 변수명 변경 부분
    # ====================
    ulta_meta = pd.DataFrame(all_ads)
    ulta_meta.to_csv(RAW_OUTPUT, index=False)

    cutoff = pd.Timestamp(datetime.now() - timedelta(days=90))
    ulta_meta["dt"] = pd.to_datetime(ulta_meta["start_date"], errors="coerce")

    ulta_meta_90 = ulta_meta[ulta_meta["dt"] >= cutoff]
    ulta_meta_90.to_csv(RAW_90D_OUTPUT, index=False)

    ulta_meta_summary = summarize(ulta_meta_90)
    ulta_meta_summary.to_csv(SUMMARY_90D_OUTPUT, index=False)

    print("완료")

    return ulta_meta, ulta_meta_90, ulta_meta_summary


def _run():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(main())
    finally:
        loop.close()

result = {}
def _thread():
    result["data"] = _run()

t = threading.Thread(target=_thread)
t.start()
t.join()

ulta_meta, ulta_meta_90, ulta_meta_summary = result["data"]

display(ulta_meta.head())
display(ulta_meta_90.head())
display(ulta_meta_summary)




[START] IT Cosmetics
[DONE] IT Cosmetics: 15
[START] medicube


Exception in thread Thread-11 (_thread):
Traceback (most recent call last):
  File "c:\Users\Jeon\anaconda3\envs\sesac_final\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\Jeon\anaconda3\envs\sesac_final\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Jeon\AppData\Local\Temp\ipykernel_35088\1131035327.py", line 224, in _thread
  File "C:\Users\Jeon\AppData\Local\Temp\ipykernel_35088\1131035327.py", line 218, in _run
  File "c:\Users\Jeon\anaconda3\envs\sesac_final\Lib\asyncio\base_events.py", line 654, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "C:\Users\Jeon\AppData\Local\Temp\ipykernel_35088\1131035327.py", line 184, in main
  File "c:\Users\Jeon\anaconda3\envs\sesac_final\Lib\site-packages\playwright\async_api\_generated.py", line 9045, in goto
    await self._impl_obj.goto(
  File "c:\Users\Jeon\anaconda3\envs\sesac_final\Lib\site-packages\playwright\_impl\_

KeyError: 'data'

### sephora top10 브랜드 메타 크롤링 (200개 크롤링 -> 최근 3개월 필터링)

In [ ]:
from playwright.async_api import async_playwright
from datetime import datetime, timedelta
from urllib.parse import quote
from pathlib import Path
import pandas as pd
import asyncio
import threading
import json
import re

INPUT_JSON = "./sephora_rankings_current.jsonl"
PAGE_ID_JSON = "./brand_page_ids_all.json"

HEADLESS = False
MAX_ADS_PER_BRAND = 200
SCROLL_ROUNDS = 40
SCROLL_WAIT_MS = 2200

RAW_OUTPUT = "./meta_crawling_result/sephora_meta.csv"
RAW_90D_OUTPUT = "./meta_crawling_result/sephora_meta_90.csv"
SUMMARY_90D_OUTPUT = "./meta_crawling_result/sephora_meta_summary.csv"


def load_product_json(path):
    ext = Path(path).suffix.lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext == ".jsonl":
            data = [json.loads(line) for line in f if line.strip()]
        else:
            data = json.load(f)
    return data


def extract_unique_brands(data):
    return list(dict.fromkeys([str(x.get("brand", "")).strip() for x in data if x.get("brand")]))


def detect_channel(path):
    name = Path(path).name.lower()
    for ch in ["ulta", "sephora", "rakuten", "qoo10"]:
        if ch in name:
            return ch
    raise ValueError("채널 감지 실패")


def load_channel_page_map(path, channel):
    config = json.load(open(path, "r", encoding="utf-8"))
    return config[channel]["market"], config[channel]["brands"]


def build_page_url(page_id, country):
    return f"https://www.facebook.com/ads/library/?active_status=active&ad_type=all&country=ALL&is_targeted_country=false&media_type=all&search_type=page&view_all_page_id={page_id}"


def normalize_text(text):
    return re.sub(r"\s+", " ", text.lower().strip()) if text else ""


def extract_date(text):
    m = re.search(r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})", text)
    if m:
        return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    return ""


def extract_text(raw):
    if not raw:
        return ""
    lines = [x.strip() for x in raw.split("\n") if x.strip()]
    for line in lines:
        if len(line) > 30 and "게재 시작" not in line and "광고" not in line:
            return line
    return ""


async def scroll(page):
    prev = 0
    for _ in range(SCROLL_ROUNDS):
        h = await page.evaluate("document.body.scrollHeight")
        if h == prev:
            break
        prev = h
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(SCROLL_WAIT_MS)


async def get_cards(page):
    return page.locator('[data-testid="ad-library-dynamic-content-container"]').locator("xpath=ancestor::div[5]")


async def parse_card(card, brand, page_id, market, retailer, mode):
    raw = ""
    try:
        raw = await card.inner_text()
    except:
        pass

    return {
        "brand": brand,
        "page_id": page_id,
        "market": market,
        "retailer": retailer,
        "source_mode": mode,
        "library_id": re.search(r"\d{10,}", raw).group(0) if re.search(r"\d{10,}", raw) else "",
        "media_type": "video" if "video" in raw.lower() else "image",
        "start_date": extract_date(raw),
        "ad_text": extract_text(raw),
    }


async def collect(page, brand, page_id, market, retailer, mode):
    await scroll(page)
    cards = await get_cards(page)
    count = await cards.count()

    ads = []
    seen = set()

    for i in range(count):
        card = cards.nth(i)
        data = await parse_card(card, brand, page_id, market, retailer, mode)

        key = (data["library_id"], normalize_text(data["ad_text"]))
        if key in seen:
            continue

        seen.add(key)
        ads.append(data)

        if len(ads) >= MAX_ADS_PER_BRAND:
            break

    return ads


def summarize(df):
    rows = []
    now = datetime.now()

    for brand, g in df.groupby("brand"):
        total = len(g)
        img = (g["media_type"] == "image").sum()
        vid = (g["media_type"] == "video").sum()

        dates = pd.to_datetime(g["start_date"], errors="coerce").dropna()
        latest = dates.max().strftime("%Y-%m-%d") if len(dates) else ""
        recent30 = (dates >= pd.Timestamp(now - timedelta(days=30))).sum()

        rows.append({
            "brand": brand,
            "total_ads": total,
            "image_ads": int(img),
            "video_ads": int(vid),
            "image_ratio": round(img/total, 3) if total else 0,
            "video_ratio": round(vid/total, 3) if total else 0,
            "latest_ad_date": latest,
            "recent_30d_ads": int(recent30),
        })

    return pd.DataFrame(rows)


async def main():
    product = load_product_json(INPUT_JSON)
    brands = extract_unique_brands(product)

    channel = detect_channel(INPUT_JSON)
    market, page_map = load_channel_page_map(PAGE_ID_JSON, channel)

    all_ads = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()

        for brand in brands:
            print(f"[START] {brand}")
            page_id = page_map[brand]["page_id"]
            await page.goto(build_page_url(page_id, market))
            await page.wait_for_timeout(3000)
            ads = await collect(page, brand, page_id, market, channel, "page")
            all_ads.extend(ads)
            print(f"[DONE] {brand}: {len(ads)}")

        await browser.close()

    sephora_meta = pd.DataFrame(all_ads)
    sephora_meta.to_csv(RAW_OUTPUT, index=False)

    cutoff = pd.Timestamp(datetime.now() - timedelta(days=90))
    sephora_meta["dt"] = pd.to_datetime(sephora_meta["start_date"], errors="coerce")

    sephora_meta_90 = sephora_meta[sephora_meta["dt"] >= cutoff]
    sephora_meta_90.to_csv(RAW_90D_OUTPUT, index=False)

    sephora_meta_summary = summarize(sephora_meta_90)
    sephora_meta_summary.to_csv(SUMMARY_90D_OUTPUT, index=False)

    print("완료")
    return sephora_meta, sephora_meta_90, sephora_meta_summary


def _run():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(main())
    finally:
        loop.close()

result = {}
def _thread():
    result["data"] = _run()

t = threading.Thread(target=_thread)
t.start()
t.join()

sephora_meta, sephora_meta_90, sephora_meta_summary = result["data"]

display(sephora_meta.head())
display(sephora_meta_90.head())
display(sephora_meta_summary)

[START] rhode
[DONE] rhode: 27
[START] The Ordinary
[DONE] The Ordinary: 26
[START] Beauty of Joseon
[DONE] Beauty of Joseon: 33
[START] EADEM
[DONE] EADEM: 10
[START] Tower 28 Beauty
[DONE] Tower 28 Beauty: 19
[START] Biodance
[FALLBACK] Biodance
[DONE] Biodance: 20
[START] Touchland
[DONE] Touchland: 200
완료


,brand,page_id,market,retailer,source_mode,library_id,media_type,start_date,ad_text,dt
0,rhode,109162541773169,US,sephora,page,801775222464319,image,2025-11-05,,2025-11-05
1,rhode,109162541773169,US,sephora,page,813148531123847,image,2025-10-20,curated skincare essentials 🤍 nourishing formu...,2025-10-20
2,rhode,109162541773169,US,sephora,page,789071247295406,image,2025-11-04,,2025-11-04
3,rhode,109162541773169,US,sephora,page,2001368310697436,image,2025-11-20,SCENTED PEPTIDE LIP TINT IN ESPRESSO. BIRTHDAY...,2025-11-20
4,rhode,109162541773169,US,sephora,page,1498351081248273,image,2026-01-13,get ready with hailey + rhode 🎀​,2026-01-13


,brand,page_id,market,retailer,source_mode,library_id,media_type,start_date,ad_text,dt
4,rhode,109162541773169,US,sephora,page,1498351081248273,image,2026-01-13,get ready with hailey + rhode 🎀​,2026-01-13
5,rhode,109162541773169,US,sephora,page,1398882635072166,image,2026-01-06,get ready with hailey + rhode 🎀​,2026-01-06
6,rhode,109162541773169,US,sephora,page,1427434232257897,image,2026-03-09,,2026-03-09
7,rhode,109162541773169,US,sephora,page,1565468324539742,image,2026-03-09,instant connection 🎀 @sarah__pidgeon for rhode.,2026-03-09
8,rhode,109162541773169,US,sephora,page,1407679217706003,image,2026-03-06,meet caffeine reset ☕️ our new our sculpting c...,2026-03-06


,brand,total_ads,image_ads,video_ads,image_ratio,video_ratio,latest_ad_date,recent_30d_ads
0,Beauty of Joseon,33,33,0,1.0,0.0,2026-03-21,22
1,Biodance,20,20,0,1.0,0.0,2026-03-20,17
2,EADEM,10,10,0,1.0,0.0,2026-03-01,8
3,The Ordinary,26,26,0,1.0,0.0,2026-03-18,26
4,Touchland,174,174,0,1.0,0.0,2026-03-16,71
5,Tower 28 Beauty,18,18,0,1.0,0.0,2026-03-21,9
6,rhode,22,22,0,1.0,0.0,2026-03-20,12


### 큐텐

In [3]:
# !python3 -m pip install playwright pandas
# !python3 -m playwright install

from playwright.async_api import async_playwright
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import json
import asyncio
import threading
import re

INPUT_JSON = "./qoo10_rankings_current.jsonl"
PAGE_ID_JSON = "./brand_page_ids_all.json"

HEADLESS = True
MAX_ADS_PER_BRAND = 200
SCROLL_ROUNDS = 40
SCROLL_WAIT_MS = 2200

RAW_OUTPUT = "./meta_crawling_result/qoo10_meta.csv"
RAW_90D_OUTPUT = "./meta_crawling_result/qoo10_meta_90.csv"
SUMMARY_90D_OUTPUT = "./meta_crawling_result/qoo10_meta_summary.csv"


def load_product_json(path):
    ext = Path(path).suffix.lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext == ".jsonl":
            data = [json.loads(line) for line in f if line.strip()]
        else:
            data = json.load(f)
    return data


def extract_unique_brands(data):
    return list(dict.fromkeys([str(x.get("shop_name", "")).strip() for x in data if x.get("shop_name")]))


def detect_channel(path):
    name = Path(path).name.lower()
    for ch in ["ulta", "sephora", "rakuten", "qoo10"]:
        if ch in name:
            return ch
    raise ValueError("채널 감지 실패")


def load_channel_page_map(path, channel):
    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)
    return config[channel]["market"], config[channel]["brands"]


def build_page_url(page_id, country):
    return f"https://www.facebook.com/ads/library/?active_status=active&ad_type=all&country=ALL&is_targeted_country=false&media_type=all&search_type=page&view_all_page_id={page_id}"



def normalize_text(text):
    return re.sub(r"\s+", " ", text.lower().strip()) if text else ""


def normalize_brand(text):
    return re.sub(r"[\s\u3000]+", "", str(text).strip().lower()) if text else ""


def extract_date(text):
    if not text:
        return ""

    patterns = [
        r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})",
        r"(\d{4})-\s*(\d{1,2})-\s*(\d{1,2})",
        r"(\d{4})/\s*(\d{1,2})/\s*(\d{1,2})",
        r"(\d{4})年\s*(\d{1,2})月\s*(\d{1,2})日",
    ]

    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    return ""


def extract_text(raw):
    if not raw:
        return ""
    lines = [x.strip() for x in raw.split("\n") if x.strip()]
    ignore = [
        "라이브러리 ID",
        "Library ID",
        "게재 시작",
        "Started running",
        "광고 상세 정보 보기",
        "See ad details",
        "플랫폼",
        "광고",
    ]
    for line in lines:
        if len(line) > 20 and not any(x.lower() in line.lower() for x in ignore):
            return line
    return ""


async def scroll(page):
    prev = -1
    for _ in range(SCROLL_ROUNDS):
        h = await page.evaluate("document.body.scrollHeight")
        if h == prev:
            break
        prev = h
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(SCROLL_WAIT_MS)


async def get_cards(page):
    return page.locator(
        "xpath=//div[contains(., '라이브러리 ID') or contains(., 'Library ID')]"
    )


async def parse_card(card, brand, page_id, market, retailer, mode):
    raw = ""
    try:
        raw = await card.inner_text(timeout=5000)
    except:
        pass

    return {
        "brand": brand,
        "page_id": page_id,
        "market": market,
        "retailer": retailer,
        "source_mode": mode,
        "library_id": re.search(r"\d{10,}", raw).group(0) if re.search(r"\d{10,}", raw) else "",
        "media_type": "video" if ("video" in raw.lower() or "동영상" in raw.lower()) else "image",
        "start_date": extract_date(raw),
        "ad_text": extract_text(raw),
    }


async def collect(page, brand, page_id, market, retailer, mode):
    await scroll(page)

    cards = await get_cards(page)
    count = await cards.count()

    ads = []
    seen = set()

    for i in range(count):
        card = cards.nth(i)
        data = await parse_card(card, brand, page_id, market, retailer, mode)

        if data["library_id"]:
            key = data["library_id"]
        else:
            key = normalize_text(data["ad_text"])[:120]

        if key in seen:
            continue

        seen.add(key)
        ads.append(data)

        if len(ads) >= MAX_ADS_PER_BRAND:
            break

    return ads


def summarize(df):
    rows = []
    now = datetime.now()

    for brand, g in df.groupby("brand"):
        total = len(g)
        img = (g["media_type"] == "image").sum()
        vid = (g["media_type"] == "video").sum()

        dates = pd.to_datetime(g["start_date"], errors="coerce").dropna()

        latest = dates.max().strftime("%Y-%m-%d") if len(dates) else ""
        recent30 = (dates >= pd.Timestamp(now - timedelta(days=30))).sum()

        rows.append({
            "brand": brand,
            "total_ads": total,
            "image_ads": int(img),
            "video_ads": int(vid),
            "image_ratio": round(img / total, 3) if total else 0,
            "video_ratio": round(vid / total, 3) if total else 0,
            "latest_ad_date": latest,
            "recent_30d_ads": int(recent30),
        })

    return pd.DataFrame(rows)


async def main():
    product = load_product_json(INPUT_JSON)
    brands = extract_unique_brands(product)

    channel = detect_channel(INPUT_JSON)
    market, page_map = load_channel_page_map(PAGE_ID_JSON, channel)

    normalized_page_map = {
        normalize_brand(k): v for k, v in page_map.items()
    }

    all_ads = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()

        for brand in brands:
            print(f"[START] {brand}")

            nb = normalize_brand(brand)
            if nb not in normalized_page_map:
                print(f"[DONE] {brand}: 0")
                continue

            page_id = normalized_page_map[nb]["page_id"]

            await page.goto(
                build_page_url(page_id, market),
                wait_until="domcontentloaded",
                timeout=60000
            )
            await page.wait_for_timeout(5000)

            ads = await collect(page, brand, page_id, market, channel, "page")

            all_ads.extend(ads)
            print(f"[DONE] {brand}: {len(ads)}")

        await browser.close()

    qoo10_meta = pd.DataFrame(all_ads, columns=[
        "brand", "page_id", "market", "retailer", "source_mode",
        "library_id", "media_type", "start_date", "ad_text"
    ])
    qoo10_meta.to_csv(RAW_OUTPUT, index=False)

    cutoff = pd.Timestamp(datetime.now() - timedelta(days=90))
    qoo10_meta["dt"] = pd.to_datetime(qoo10_meta["start_date"], errors="coerce")

    qoo10_meta_90 = qoo10_meta[qoo10_meta["dt"] >= cutoff]
    qoo10_meta_90.to_csv(RAW_90D_OUTPUT, index=False)

    qoo10_meta_summary = summarize(qoo10_meta_90)
    qoo10_meta_summary.to_csv(SUMMARY_90D_OUTPUT, index=False)

    print("완료")

    return qoo10_meta, qoo10_meta_90, qoo10_meta_summary


def _run():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(main())
    finally:
        loop.close()

result = {}
def _thread():
    result["data"] = _run()

t = threading.Thread(target=_thread)
t.start()
t.join()

qoo10_meta, qoo10_meta_90, qoo10_meta_summary = result["data"]

display(qoo10_meta.head())
display(qoo10_meta_90.head())
display(qoo10_meta_summary)

[START] DOPAMY
[DONE] DOPAMY: 0
[START] be bare
[DONE] be bare: 1
[START] ザツールラボ
[DONE] ザツールラボ: 20
[START] VTコスメティックス
[DONE] VTコスメティックス: 30
완료


,brand,page_id,market,retailer,source_mode,library_id,media_type,start_date,ad_text,dt
0,be bare,988277981041744,JP,qoo10,page,2124129214989601,image,2026-03-17,毎週火曜日から、対象販売店にて1週間限定の30%OFFセールを実施中🩷,2026-03-17
1,ザツールラボ,761007497366639,JP,qoo10,page,4323215791269161,image,2026-02-03,[더툴랩 라운지 ONLY 무료 세척 서비스] 묵은 브러쉬도 '더툴랩 브러쉬 세탁소'...,2026-02-03
2,ザツールラボ,761007497366639,JP,qoo10,page,26244466861882229,image,2026-03-06,"청담샵 영업비밀, 연장모 안심 세럼카라💕",2026-03-06
3,ザツールラボ,761007497366639,JP,qoo10,page,1667270354467307,image,2026-03-07,[청담샵 영업기밀💕]프로의 화잘먹 베이스 루틴,2026-03-07
4,ザツールラボ,761007497366639,JP,qoo10,page,1335251741987650,image,2026-03-07,"청담샵 영업비밀, 연장모 안심 세럼카라💕",2026-03-07


,brand,page_id,market,retailer,source_mode,library_id,media_type,start_date,ad_text,dt
0,be bare,988277981041744,JP,qoo10,page,2124129214989601,image,2026-03-17,毎週火曜日から、対象販売店にて1週間限定の30%OFFセールを実施中🩷,2026-03-17
1,ザツールラボ,761007497366639,JP,qoo10,page,4323215791269161,image,2026-02-03,[더툴랩 라운지 ONLY 무료 세척 서비스] 묵은 브러쉬도 '더툴랩 브러쉬 세탁소'...,2026-02-03
2,ザツールラボ,761007497366639,JP,qoo10,page,26244466861882229,image,2026-03-06,"청담샵 영업비밀, 연장모 안심 세럼카라💕",2026-03-06
3,ザツールラボ,761007497366639,JP,qoo10,page,1667270354467307,image,2026-03-07,[청담샵 영업기밀💕]프로의 화잘먹 베이스 루틴,2026-03-07
4,ザツールラボ,761007497366639,JP,qoo10,page,1335251741987650,image,2026-03-07,"청담샵 영업비밀, 연장모 안심 세럼카라💕",2026-03-07


,brand,total_ads,image_ads,video_ads,image_ratio,video_ratio,latest_ad_date,recent_30d_ads
0,VTコスメティックス,30,30,0,1.0,0.0,2026-03-22,30
1,be bare,1,1,0,1.0,0.0,2026-03-17,1
2,ザツールラボ,20,20,0,1.0,0.0,2026-03-22,17


### 라쿠텐

In [4]:
# !python3 -m pip install playwright pandas
# !python3 -m playwright install

from playwright.async_api import async_playwright
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import json
import asyncio
import threading
import re

INPUT_JSON = "./rakuten_rankings_current.jsonl"
PAGE_ID_JSON = "./brand_page_ids_all.json"

HEADLESS = True
MAX_ADS_PER_BRAND = 200
SCROLL_ROUNDS = 40
SCROLL_WAIT_MS = 2200

RAW_OUTPUT = "./meta_crawling_result/rakuten_meta.csv"
RAW_90D_OUTPUT = "./meta_crawling_result/rakuten_meta_90.csv"
SUMMARY_90D_OUTPUT = "./meta_crawling_result/rakuten_meta_summary.csv"


def load_product_json(path):
    ext = Path(path).suffix.lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext == ".jsonl":
            data = [json.loads(line) for line in f if line.strip()]
        else:
            data = json.load(f)
    return data


def extract_unique_brands(data):
    return list(
        dict.fromkeys(
            [str(x.get("shop_name", "")).strip() for x in data if x.get("shop_name")]
        )
    )


def detect_channel(path):
    name = Path(path).name.lower()
    for ch in ["ulta", "sephora", "rakuten", "qoo10"]:
        if ch in name:
            return ch
    raise ValueError("채널 감지 실패")


def load_channel_page_map(path, channel):
    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)
    return config[channel]["market"], config[channel]["brands"]


def build_page_url(page_id, country):
    return f"https://www.facebook.com/ads/library/?active_status=active&ad_type=all&country=ALL&is_targeted_country=false&media_type=all&search_type=page&view_all_page_id={page_id}"



def normalize_text(text):
    return re.sub(r"\s+", " ", text.lower().strip()) if text else ""


def normalize_brand(text):
    return re.sub(r"[\s\u3000]+", "", str(text).strip().lower()) if text else ""


def extract_date(text):
    if not text:
        return ""

    patterns = [
        r"(\d{4})\.\s*(\d{1,2})\.\s*(\d{1,2})",
        r"(\d{4})-\s*(\d{1,2})-\s*(\d{1,2})",
        r"(\d{4})/\s*(\d{1,2})/\s*(\d{1,2})",
        r"(\d{4})年\s*(\d{1,2})月\s*(\d{1,2})日",
    ]

    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    return ""


def extract_text(raw):
    if not raw:
        return ""
    lines = [x.strip() for x in raw.split("\n") if x.strip()]
    ignore = [
        "라이브러리 ID",
        "Library ID",
        "게재 시작",
        "Started running",
        "광고 상세 정보 보기",
        "See ad details",
        "플랫폼",
        "광고",
    ]
    for line in lines:
        if len(line) > 20 and not any(x.lower() in line.lower() for x in ignore):
            return line
    return ""


async def scroll(page):
    prev = -1
    for _ in range(SCROLL_ROUNDS):
        h = await page.evaluate("document.body.scrollHeight")
        if h == prev:
            break
        prev = h
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(SCROLL_WAIT_MS)


async def get_cards(page):
    return page.locator(
        "xpath=//div[contains(., '라이브러리 ID') or contains(., 'Library ID')]"
    )


async def parse_card(card, brand, page_id, market, retailer, mode):
    raw = ""
    try:
        raw = await card.inner_text(timeout=5000)
    except:
        pass

    return {
        "brand": brand,
        "page_id": page_id,
        "market": market,
        "retailer": retailer,
        "source_mode": mode,
        "library_id": re.search(r"\d{10,}", raw).group(0) if re.search(r"\d{10,}", raw) else "",
        "media_type": "video" if ("video" in raw.lower() or "동영상" in raw.lower()) else "image",
        "start_date": extract_date(raw),
        "ad_text": extract_text(raw),
    }


async def collect(page, brand, page_id, market, retailer, mode):
    await scroll(page)

    cards = await get_cards(page)
    count = await cards.count()

    ads = []
    seen = set()

    for i in range(count):
        card = cards.nth(i)
        data = await parse_card(card, brand, page_id, market, retailer, mode)

        if data["library_id"]:
            key = data["library_id"]
        else:
            key = normalize_text(data["ad_text"])[:120]

        if key in seen:
            continue

        seen.add(key)
        ads.append(data)

        if len(ads) >= MAX_ADS_PER_BRAND:
            break

    return ads


def summarize(df):
    rows = []
    now = datetime.now()

    for brand, g in df.groupby("brand"):
        total = len(g)
        img = (g["media_type"] == "image").sum()
        vid = (g["media_type"] == "video").sum()

        dates = pd.to_datetime(g["start_date"], errors="coerce").dropna()

        latest = dates.max().strftime("%Y-%m-%d") if len(dates) else ""
        recent30 = (dates >= pd.Timestamp(now - timedelta(days=30))).sum()

        rows.append({
            "brand": brand,
            "total_ads": total,
            "image_ads": int(img),
            "video_ads": int(vid),
            "image_ratio": round(img / total, 3) if total else 0,
            "video_ratio": round(vid / total, 3) if total else 0,
            "latest_ad_date": latest,
            "recent_30d_ads": int(recent30),
        })

    return pd.DataFrame(rows)


async def main():
    product = load_product_json(INPUT_JSON)
    brands = extract_unique_brands(product)

    channel = detect_channel(INPUT_JSON)
    market, page_map = load_channel_page_map(PAGE_ID_JSON, channel)

    normalized_page_map = {
        normalize_brand(k): v for k, v in page_map.items()
    }

    all_ads = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()

        for brand in brands:
            print(f"[START] {brand}")

            nb = normalize_brand(brand)
            if nb not in normalized_page_map:
                print(f"[DONE] {brand}: 0")
                continue

            page_id = normalized_page_map[nb]["page_id"]

            await page.goto(
                build_page_url(page_id, market),
                wait_until="domcontentloaded",
                timeout=60000
            )
            await page.wait_for_timeout(5000)

            ads = await collect(page, brand, page_id, market, channel, "page")

            all_ads.extend(ads)
            print(f"[DONE] {brand}: {len(ads)}")

        await browser.close()

    rakuten_meta = pd.DataFrame(all_ads, columns=[
        "brand", "page_id", "market", "retailer", "source_mode",
        "library_id", "media_type", "start_date", "ad_text"
    ])
    rakuten_meta.to_csv(RAW_OUTPUT, index=False)

    cutoff = pd.Timestamp(datetime.now() - timedelta(days=90))
    rakuten_meta["dt"] = pd.to_datetime(rakuten_meta["start_date"], errors="coerce")

    rakuten_meta_90 = rakuten_meta[rakuten_meta["dt"] >= cutoff]
    rakuten_meta_90.to_csv(RAW_90D_OUTPUT, index=False)

    rakuten_meta_summary = summarize(rakuten_meta_90)
    rakuten_meta_summary.to_csv(SUMMARY_90D_OUTPUT, index=False)

    print("완료")

    return rakuten_meta, rakuten_meta_90, rakuten_meta_summary


def _run():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(main())
    finally:
        loop.close()

result = {}
def _thread():
    result["data"] = _run()

t = threading.Thread(target=_thread)
t.start()
t.join()

rakuten_meta, rakuten_meta_90, rakuten_meta_summary = result["data"]

display(rakuten_meta.head())
display(rakuten_meta_90.head())
display(rakuten_meta_summary)

[START] VTcosmetic楽天市場店
[DONE] VTcosmetic楽天市場店: 30
[START] アテニア公式ショップ　楽天市場店
[DONE] アテニア公式ショップ　楽天市場店: 24
[START] 【公式】Yunth Store 楽天市場店
[DONE] 【公式】Yunth Store 楽天市場店: 0
[START] コスメティック　やよい
[DONE] コスメティック　やよい: 0
[START] ビタミンC誘導体のトゥヴェール
[DONE] ビタミンC誘導体のトゥヴェール: 6
[START] シュウ ウエムラ 公式ショップ
[DONE] シュウ ウエムラ 公式ショップ: 0
[START] SK-II 公式ショップ楽天市場店
[DONE] SK-II 公式ショップ楽天市場店: 22
[START] FANCL公式ショップ 楽天市場店
[DONE] FANCL公式ショップ 楽天市場店: 12
완료


,brand,page_id,market,retailer,source_mode,library_id,media_type,start_date,ad_text,dt
0,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,2013114266298182,image,2026-03-12,marie_nekomimi 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-12
1,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,919007000943662,image,2026-03-19,望月拓未 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19
2,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,954023060479838,image,2026-03-19,marie_nekomimi 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19
3,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,4450413355191474,image,2026-03-19,marie_nekomimi 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19
4,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,922881477269821,image,2026-03-19,Insta用ページ 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19


,brand,page_id,market,retailer,source_mode,library_id,media_type,start_date,ad_text,dt
0,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,2013114266298182,image,2026-03-12,marie_nekomimi 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-12
1,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,919007000943662,image,2026-03-19,望月拓未 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19
2,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,954023060479838,image,2026-03-19,marie_nekomimi 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19
3,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,4450413355191474,image,2026-03-19,marie_nekomimi 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19
4,VTcosmetic楽天市場店,106232352197012,JP,rakuten,page,922881477269821,image,2026-03-19,Insta用ページ 페이지는 VT cosmetic JP과(와) 함께합니다,2026-03-19


,brand,total_ads,image_ads,video_ads,image_ratio,video_ratio,latest_ad_date,recent_30d_ads
0,FANCL公式ショップ 楽天市場店,12,12,0,1.000,0.000,2026-03-19,12
1,SK-II 公式ショップ楽天市場店,22,22,0,1.000,0.000,2026-03-15,22
2,VTcosmetic楽天市場店,30,30,0,1.000,0.000,2026-03-22,30
3,アテニア公式ショップ 楽天市場店,24,21,3,0.875,0.125,2026-03-19,21
4,ビタミンC誘導体のトゥヴェール,6,6,0,1.000,0.000,2026-03-19,6
